In [1]:
from utility.adjust_redshift import adjust_redshift

from sparcl.client import SparclClient

import numpy as np
import pandas as pd
import json
import h5py
import time

property_file_path = "data/target_properties.csv"
spectra_hdf5_path = "data/batches/spectra.h5"

In [2]:
df = pd.read_csv(property_file_path,index_col=0)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 404142 entries, 0 to 404141
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   targetid            404142 non-null  int64  
 1   z                   404142 non-null  float64
 2   zerr                404142 non-null  float64
 3   civ_1549_flux       404142 non-null  float64
 4   civ_1549_flux_ivar  404142 non-null  float64
 5   civ_1549_sigma      404142 non-null  float64
 6   flux_w3             404142 non-null  float64
 7   flux_ivar_w3        403793 non-null  float64
 8   desiname            404142 non-null  object 
dtypes: float64(7), int64(1), object(1)
memory usage: 30.8+ MB


In [3]:
# store the duplicate targets
duplicate_targetidxs = []

In [ ]:
# detect duplicates, add to list, then remove from dataframe
duplicates = df[df["targetid"].duplicated()]["targetid"].unique()
# duplicate_targetidxs.append(duplicates)  # already done
df = df.drop_duplicates(subset="targetid",keep=False)

In [5]:
target_idxs = [int(x) for x in df["targetid"]]

n = len(target_idxs)
batch_size = 500 # sparcl cannot take more than 500 idxs at once 
n_batches = int((n / batch_size)+1)

print(f"{batch_size=}, {n_batches=}")

batch_size=500, n_batches=764


In [6]:
client = SparclClient()

announcement=Data set deprecation notice: on November 19, 2025 the SDSS/BOSS DR16 data sets were deprecated. Please use the new SDSS/BOSS DR17 data sets instead.


In [7]:
output_fields = ['targetid', 'redshift', 'wavelength', 'flux', 'ivar']

# results = client.retrieve_by_specid(specid_list=test_idxs, include=output_fields, dataset_list=["DESI-DR1"],limit=None,fmt="pandas")

In [ ]:
# duplicates = results[results["targetid"].duplicated()]["targetid"].unique()
# duplicate_targetidxs.append(duplicates)
# results = results.drop_duplicates(subset="targetid",keep=False)

In [ ]:
# for idx in results.index:
#     lam, flux, ivar = adjust_redshift(results.loc[idx,"redshift"], results.loc[idx,"wavelength"], results.loc[idx,"flux"], results.loc[idx,"ivar"])

#     valid_pixel_mask = ((lam > 1150) & (lam < 1810)) & (ivar > 0) & np.isfinite(ivar) & (flux != 0)

#     arr = np.array([lam[valid_pixel_mask],flux[valid_pixel_mask], ivar[valid_pixel_mask]])

#     with h5py.File("spectra.h5", "a") as f: 
#         dset = f.create_dataset(  
#             f"{results.loc[idx,'targetid']}", data=arr  
#         )



In [45]:
for i in range(40,60):
    tic = time.time()
    print(f"Start batch {i}")

    # get next 500 targetids
    curr_idxs = target_idxs[batch_size*i:batch_size*(i+1)]

    # retrieve spectra
    results = client.retrieve_by_specid(specid_list=curr_idxs, include=output_fields, dataset_list=["DESI-DR1"],limit=None,fmt="pandas")

    # detect duplicates, add to duplicate list, then remove them
    duplicates = results[results["targetid"].duplicated()]["targetid"].unique()
    duplicate_targetidxs.extend(duplicates)
    results = results.drop_duplicates(subset="targetid",keep=False)

    # redshift correct and truncate each spectrum to relevant window, then save as (3,N) numpy array to HDF5 file
    for idx in results.index:
        lam, flux, ivar = adjust_redshift(results.loc[idx,"redshift"], results.loc[idx,"wavelength"], results.loc[idx,"flux"], results.loc[idx,"ivar"])

        valid_pixel_mask = ((lam > 1150) & (lam < 1810)) & (ivar > 0) & np.isfinite(ivar) & (flux != 0)

        arr = np.array([lam[valid_pixel_mask],flux[valid_pixel_mask], ivar[valid_pixel_mask]])

        with h5py.File(spectra_hdf5_path, "a") as f: 
            dset = f.create_dataset(  
                f"{results.loc[idx,'targetid']}", data=arr  
            )
    
    toc = time.time() - tic
    print(f"Batch {i} finished in {toc:.2f}s\n")


Start batch 40
Batch 40 finished in 22.89s

Start batch 41
Batch 41 finished in 19.34s

Start batch 42
Batch 42 finished in 26.60s

Start batch 43
Batch 43 finished in 21.82s

Start batch 44
Batch 44 finished in 39.32s

Start batch 45
Batch 45 finished in 21.24s

Start batch 46
Batch 46 finished in 57.03s

Start batch 47
Batch 47 finished in 21.25s

Start batch 48
Batch 48 finished in 32.69s

Start batch 49
Batch 49 finished in 33.10s

Start batch 50
Batch 50 finished in 20.13s

Start batch 51
Batch 51 finished in 19.27s

Start batch 52
Batch 52 finished in 20.17s

Start batch 53
Batch 53 finished in 21.05s

Start batch 54
Batch 54 finished in 21.31s

Start batch 55
Batch 55 finished in 20.60s

Start batch 56
Batch 56 finished in 18.35s

Start batch 57
Batch 57 finished in 22.38s

Start batch 58
Batch 58 finished in 23.75s

Start batch 59
Batch 59 finished in 39.81s



In [46]:
with h5py.File(spectra_hdf5_path, "r") as f:
    # test = f[f"{results.loc[400,'targetid']}"][()]
    print(len(set(f.keys())))

29920


In [48]:
with open("data/batches/duplicate_targets.txt","a") as f:
    f.write(str(duplicate_targetidxs))